In [19]:
import pandas as pd

df = pd.read_csv('Cleaned_delayed_flights.csv')

# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])

print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min()} → {df['Date'].max()}")
df.info()

Shape: (1928368, 33)
Date range: 2008-01-01 00:00:00 → 2008-12-31 00:00:00
<class 'pandas.DataFrame'>
RangeIndex: 1928368 entries, 0 to 1928367
Data columns (total 33 columns):
 #   Column                Dtype         
---  ------                -----         
 0   Date                  datetime64[us]
 1   Year                  int64         
 2   Month_Name            str           
 3   Month                 int64         
 4   Day                   int64         
 5   Day_Name              str           
 6   DayOfWeek_Egypt       int64         
 7   UniqueCarrier         str           
 8   FlightNum             int64         
 9   TailNum               str           
 10  Origin                str           
 11  Dest                  str           
 12  Distance              int64         
 13  Scheduled_DepTime     str           
 14  Actual_DepTime        str           
 15  Departure_Delay       float64       
 16  Scheduled_ArrTime     str           
 17  Actual_ArrTime      

In [20]:
#1. Dim_date
def create_date_dimension(start_date, end_date):
    """Generate a full Date dimension table between two dates."""
    dates = pd.date_range(start=start_date, end=end_date)
    dim = pd.DataFrame(dates, columns=['FullDate'])

    dim['DateKey']    = dim['FullDate'].dt.strftime('%Y%m%d').astype(int)
    dim['Year']       = dim['FullDate'].dt.year
    dim['Month']      = dim['FullDate'].dt.month
    dim['Day']        = dim['FullDate'].dt.day
    dim['Quarter']    = dim['FullDate'].dt.quarter
    dim['DayName']    = dim['FullDate'].dt.day_name()
    dim['MonthName']  = dim['FullDate'].dt.month_name()
    dim['DayOfWeek']  = dim['FullDate'].dt.dayofweek   # 0 = Monday, 6 = Sunday
    dim['IsWeekend']  = dim['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)
    dim['WeekOfYear'] = dim['FullDate'].dt.isocalendar().week

    return dim

# Build for the year covered by our data
dim_date = create_date_dimension('2008-01-01', '2008-12-31')

# Rename to match merge key used later
dim_date = dim_date.rename(columns={'FullDate': 'Date'})

print(dim_date.shape)
dim_date.head()

(366, 11)


,Date,DateKey,Year,Month,Day,Quarter,DayName,MonthName,DayOfWeek,IsWeekend,WeekOfYear
0,2008-01-01,20080101,2008,1,1,1,Tuesday,January,1,0,1
1,2008-01-02,20080102,2008,1,2,1,Wednesday,January,2,0,1
2,2008-01-03,20080103,2008,1,3,1,Thursday,January,3,0,1
3,2008-01-04,20080104,2008,1,4,1,Friday,January,4,0,1
4,2008-01-05,20080105,2008,1,5,1,Saturday,January,5,1,1


In [21]:
# 2. Dim_Airport
all_airports = pd.concat([df['Origin'], df['Dest']]).unique()
dim_airport = pd.DataFrame(all_airports, columns=['AirportCode'])
dim_airport['airport_FK'] = range(1, len(dim_airport) + 1)
# Reorder columns
dim_airport = dim_airport[['airport_FK', 'AirportCode']]

dim_airport.head()

,airport_FK,AirportCode
0,1,IAD
1,2,IND
2,3,ISP
3,4,JAN
4,5,JAX


In [22]:
# 3. Dim_carrier
carrier_names = {
    "WN": "Southwest Airlines",
    "XE": "ExpressJet Airlines",
    "YV": "Mesa Airlines",
    "OH": "PSA Airlines",
    "OO": "SkyWest Airlines",
    "UA": "United Airlines",
    "US": "US Airways",
    "DL": "Delta Air Lines",
    "EV": "Atlantic Southeast Airlines",
    "F9": "Frontier Airlines",
    "FL": "AirTran Airways",
    "HA": "Hawaiian Airlines",
    "MQ": "American Eagle Airlines",
    "NW": "Northwest Airlines",
    "9E": "Pinnacle Airlines",
    "AA": "American Airlines",
    "AQ": "Aloha Airlines",
    "AS": "Alaska Airlines",
    "B6": "JetBlue Airways",
    "CO": "Continental Airlines"
}

dim_carrier = pd.DataFrame(list(carrier_names.items()), columns=['UniqueCarrier', 'CarrierName'])

# Add surrogate key
dim_carrier['CarrierKey'] = range(1, len(dim_carrier) + 1)

# Reorder columns
dim_carrier = dim_carrier[['CarrierKey', 'UniqueCarrier', 'CarrierName']]

print(dim_carrier.shape)
dim_carrier

(20, 3)


,CarrierKey,UniqueCarrier,CarrierName
0,1,WN,Southwest Airlines
1,2,XE,ExpressJet Airlines
2,3,YV,Mesa Airlines
3,4,OH,PSA Airlines
4,5,OO,SkyWest Airlines
5,6,UA,United Airlines
6,7,US,US Airways
7,8,DL,Delta Air Lines
8,9,EV,Atlantic Southeast Airlines
9,10,F9,Frontier Airlines


In [23]:
# 4. Dim_Aircraft 
dim_aircraft = df[['TailNum']].drop_duplicates().reset_index(drop=True)
dim_aircraft.columns = ['TailNum']
dim_aircraft['aircraft_FK'] = range(1, len(dim_aircraft) + 1)

dim_aircraft.head()

,TailNum,aircraft_FK
0,N712SW,1
1,N772SW,2
2,N428WN,3
3,N464WN,4
4,N726SW,5


In [24]:
#5.Dim_route
# Extract unique Origin–Dest–Distance combinations from the raw data
dim_route = df[['Origin', 'Dest', 'Distance']].drop_duplicates().reset_index(drop=True).copy()

# Add surrogate key
dim_route['route_FK'] = range(1, len(dim_route) + 1)

# Reorder columns
dim_route = dim_route[['route_FK', 'Origin', 'Dest', 'Distance']]

print(dim_route.shape)
dim_route.head()

(5127, 4)


,route_FK,Origin,Dest,Distance
0,1,IAD,TPA,810
1,2,IND,BWI,515
2,3,IND,JAX,688
3,4,IND,LAS,1591
4,5,IND,MCO,828


In [25]:
#5.Dim_time
# 1. Create a range of times for every minute of the day
time_range = pd.date_range("00:00", "23:59", freq="min").time
dim_time = pd.DataFrame({"Time": time_range})

# 2. Add Surrogate Key (TimeKey) in HHMM format
dim_time['TimeKey'] = dim_time['Time'].apply(lambda x: int(x.strftime('%H%M')))

# 3. Add useful attributes like Hour and Period of Day
dim_time['Hour'] = dim_time['Time'].apply(lambda x: x.hour)
dim_time['Minute'] = dim_time['Time'].apply(lambda x: x.minute)

# Categorize time into periods
def get_period(h):
    if 5 <= h < 12: return 'Morning'
    elif 12 <= h < 17: return 'Afternoon'
    elif 17 <= h < 21: return 'Evening'
    else: return 'Night'

dim_time['Period'] = dim_time['Hour'].apply(get_period)

# Reorder columns
dim_time = dim_time[['TimeKey', 'Time', 'Hour', 'Minute', 'Period']]

print("Dim_Time created successfully!")
dim_time.head()

Dim_Time created successfully!


,TimeKey,Time,Hour,Minute,Period
0,0,00:00:00,0,0,Night
1,1,00:01:00,0,1,Night
2,2,00:02:00,0,2,Night
3,3,00:03:00,0,3,Night
4,4,00:04:00,0,4,Night


In [26]:
#merge Dim_Date with fact table
fact = df.merge(dim_date[['Date', 'DateKey']], on='Date', how='left')

# Drop raw date columns now captured by DateKey
fact = fact.drop(columns=['Date', 'Year', 'Month', 'Day', 'DayOfWeek_Egypt'])

print(fact.shape)
fact.head(2)

(1928368, 29)


,Month_Name,Day_Name,UniqueCarrier,FlightNum,TailNum,Origin,Dest,Distance,Scheduled_DepTime,Actual_DepTime,...,Delay_sum,ActualElapsedTime,Expected_ElapsedTime,AirTime,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,DateKey
0,January,Thursday,WN,335,N712SW,IAD,TPA,810,19:55:00,20:03:00,...,0.0,128.0,150.0,116.0,4.0,8.0,0,N,0,20080103
1,January,Thursday,WN,3231,N772SW,IAD,TPA,810,07:35:00,07:54:00,...,0.0,128.0,145.0,113.0,5.0,10.0,0,N,0,20080103


In [27]:
#merge Dim_Airport with fact table
fact = fact.merge(
    dim_airport[['airport_FK', 'AirportCode']],
    left_on='Origin', right_on='AirportCode', how='left'
)
fact = fact.rename(columns={'airport_FK': 'Origin_FK'})
fact = fact.drop(columns=['AirportCode'])

print(fact.shape)
fact.head()

(1928368, 30)


,Month_Name,Day_Name,UniqueCarrier,FlightNum,TailNum,Origin,Dest,Distance,Scheduled_DepTime,Actual_DepTime,...,ActualElapsedTime,Expected_ElapsedTime,AirTime,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,DateKey,Origin_FK
0,January,Thursday,WN,335,N712SW,IAD,TPA,810,19:55:00,20:03:00,...,128.0,150.0,116.0,4.0,8.0,0,N,0,20080103,1
1,January,Thursday,WN,3231,N772SW,IAD,TPA,810,07:35:00,07:54:00,...,128.0,145.0,113.0,5.0,10.0,0,N,0,20080103,1
2,January,Thursday,WN,448,N428WN,IND,BWI,515,06:20:00,06:28:00,...,96.0,90.0,76.0,3.0,17.0,0,N,0,20080103,2
3,January,Thursday,WN,3920,N464WN,IND,BWI,515,17:55:00,18:29:00,...,90.0,90.0,77.0,3.0,10.0,0,N,0,20080103,2
4,January,Thursday,WN,378,N726SW,IND,JAX,688,19:15:00,19:40:00,...,101.0,115.0,87.0,4.0,10.0,0,N,0,20080103,2


In [28]:
#merge Dim_Airport with fact table
fact = fact.merge(
    dim_airport[['airport_FK', 'AirportCode']],
    left_on='Dest', right_on='AirportCode', how='left'
)
fact = fact.rename(columns={'airport_FK': 'Dest_FK'})
fact = fact.drop(columns=['AirportCode'])

print(fact.shape)
fact.head()

(1928368, 31)


,Month_Name,Day_Name,UniqueCarrier,FlightNum,TailNum,Origin,Dest,Distance,Scheduled_DepTime,Actual_DepTime,...,Expected_ElapsedTime,AirTime,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,DateKey,Origin_FK,Dest_FK
0,January,Thursday,WN,335,N712SW,IAD,TPA,810,19:55:00,20:03:00,...,150.0,116.0,4.0,8.0,0,N,0,20080103,1,40
1,January,Thursday,WN,3231,N772SW,IAD,TPA,810,07:35:00,07:54:00,...,145.0,113.0,5.0,10.0,0,N,0,20080103,1,40
2,January,Thursday,WN,448,N428WN,IND,BWI,515,06:20:00,06:28:00,...,90.0,76.0,3.0,17.0,0,N,0,20080103,2,53
3,January,Thursday,WN,3920,N464WN,IND,BWI,515,17:55:00,18:29:00,...,90.0,77.0,3.0,10.0,0,N,0,20080103,2,53
4,January,Thursday,WN,378,N726SW,IND,JAX,688,19:15:00,19:40:00,...,115.0,87.0,4.0,10.0,0,N,0,20080103,2,5


In [29]:
#merge Dim_Aircraft with fact table
fact = fact.merge(
    dim_aircraft[['aircraft_FK', 'TailNum']],
    on='TailNum', how='left'
)
fact = fact.drop(columns=['TailNum'])

print(fact.shape)

(1928368, 31)


In [30]:
#merge Dim_Route with fact table

fact = fact.merge(dim_route[['route_FK', 'Origin', 'Dest']], on=['Origin', 'Dest'], how='left')
fact = fact.drop(columns=['Origin', 'Dest'], errors='ignore')

print(fact.shape)
fact.head()

(1928368, 30)


,Month_Name,Day_Name,UniqueCarrier,FlightNum,Distance,Scheduled_DepTime,Actual_DepTime,Departure_Delay,Scheduled_ArrTime,Actual_ArrTime,...,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,DateKey,Origin_FK,Dest_FK,aircraft_FK,route_FK
0,January,Thursday,WN,335,810,19:55:00,20:03:00,8.0,22:25:00,22:11:00,...,4.0,8.0,0,N,0,20080103,1,40,1,1
1,January,Thursday,WN,3231,810,07:35:00,07:54:00,19.0,10:00:00,10:02:00,...,5.0,10.0,0,N,0,20080103,1,40,2,1
2,January,Thursday,WN,448,515,06:20:00,06:28:00,8.0,07:50:00,08:04:00,...,3.0,17.0,0,N,0,20080103,2,53,3,2
3,January,Thursday,WN,3920,515,17:55:00,18:29:00,34.0,19:25:00,19:59:00,...,3.0,10.0,0,N,0,20080103,2,53,4,2
4,January,Thursday,WN,378,688,19:15:00,19:40:00,25.0,21:10:00,21:21:00,...,4.0,10.0,0,N,0,20080103,2,5,5,3


In [31]:
#merge dim_carrier with fact table
fact = fact.merge(
    dim_carrier[['CarrierKey', 'UniqueCarrier']],
    on='UniqueCarrier', how='left'
)
fact = fact.drop(columns=['UniqueCarrier'])

print(fact.shape)
fact.head(3)

(1928368, 30)


,Month_Name,Day_Name,FlightNum,Distance,Scheduled_DepTime,Actual_DepTime,Departure_Delay,Scheduled_ArrTime,Actual_ArrTime,Arrival_Delay,...,TaxiOut,Cancelled,CancellationCode,Diverted,DateKey,Origin_FK,Dest_FK,aircraft_FK,route_FK,CarrierKey
0,January,Thursday,335,810,19:55:00,20:03:00,8.0,22:25:00,22:11:00,-14.0,...,8.0,0,N,0,20080103,1,40,1,1,1
1,January,Thursday,3231,810,07:35:00,07:54:00,19.0,10:00:00,10:02:00,2.0,...,10.0,0,N,0,20080103,1,40,2,1,1
2,January,Thursday,448,515,06:20:00,06:28:00,8.0,07:50:00,08:04:00,14.0,...,17.0,0,N,0,20080103,2,53,3,2,1


In [32]:
# Final Fact table reviw
print("Fact table columns:")
print(fact.columns.tolist())
print()
print(fact.dtypes)
print()
print(f"Missing values:\n{fact.isnull().sum()[fact.isnull().sum() > 0]}")

Fact table columns:
['Month_Name', 'Day_Name', 'FlightNum', 'Distance', 'Scheduled_DepTime', 'Actual_DepTime', 'Departure_Delay', 'Scheduled_ArrTime', 'Actual_ArrTime', 'Arrival_Delay', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'Delay_sum', 'ActualElapsedTime', 'Expected_ElapsedTime', 'AirTime', 'TaxiIn', 'TaxiOut', 'Cancelled', 'CancellationCode', 'Diverted', 'DateKey', 'Origin_FK', 'Dest_FK', 'aircraft_FK', 'route_FK', 'CarrierKey']

Month_Name                  str
Day_Name                    str
FlightNum                 int64
Distance                  int64
Scheduled_DepTime           str
Actual_DepTime              str
Departure_Delay         float64
Scheduled_ArrTime           str
Actual_ArrTime              str
Arrival_Delay           float64
CarrierDelay            float64
WeatherDelay            float64
NASDelay                float64
SecurityDelay           float64
LateAircraftDelay       float64
Delay_sum               float64
ActualEl

In [33]:
import pandas as pd
import numpy as np

# A. Create Dim_Time
time_range = pd.date_range("00:00", "23:59", freq="min").time
dim_time = pd.DataFrame({"Time": time_range})
dim_time['TimeKey'] = dim_time['Time'].apply(lambda x: int(x.strftime('%H%M')))
dim_time['Hour'] = dim_time['Time'].apply(lambda x: x.hour)

def get_period(h):
    if 5 <= h < 12: return 'Morning'
    elif 12 <= h < 17: return 'Afternoon'
    elif 17 <= h < 21: return 'Evening'
    else: return 'Night'

dim_time['Period'] = dim_time['Hour'].apply(get_period)

# B. Fast vectorized conversion (بدل apply)
time_columns = ['Scheduled_DepTime', 'Actual_DepTime', 'Scheduled_ArrTime', 'Actual_ArrTime']

for col in time_columns:
    if col in fact.columns:
        t = pd.to_datetime(fact[col], errors='coerce')
        fact[f'{col}_Key'] = (t.dt.hour * 100 + t.dt.minute).where(t.notna(), np.nan)

# C. Drop original columns
fact = fact.drop(columns=time_columns, errors='ignore')

print("Done! Fact shape:", fact.shape)
fact.head(3)


C:\Users\Rokaya Samy\AppData\Local\Temp\ipykernel_37948\2051239938.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  t = pd.to_datetime(fact[col], errors='coerce')
C:\Users\Rokaya Samy\AppData\Local\Temp\ipykernel_37948\2051239938.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  t = pd.to_datetime(fact[col], errors='coerce')
C:\Users\Rokaya Samy\AppData\Local\Temp\ipykernel_37948\2051239938.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  t = pd.to_datetime(fact[col], errors='coerce')
C:\Users\Rokaya Samy\AppData\Local\Temp\ipykernel_37948\2051239938.py:23: UserW

Done! Fact shape: (1928368, 30)


,Month_Name,Day_Name,FlightNum,Distance,Departure_Delay,Arrival_Delay,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,...,DateKey,Origin_FK,Dest_FK,aircraft_FK,route_FK,CarrierKey,Scheduled_DepTime_Key,Actual_DepTime_Key,Scheduled_ArrTime_Key,Actual_ArrTime_Key
0,January,Thursday,335,810,8.0,-14.0,0.0,0.0,0.0,0.0,...,20080103,1,40,1,1,1,1955,2003,2225,2211
1,January,Thursday,3231,810,19.0,2.0,0.0,0.0,0.0,0.0,...,20080103,1,40,2,1,1,735,754,1000,1002
2,January,Thursday,448,515,8.0,14.0,0.0,0.0,0.0,0.0,...,20080103,2,53,3,2,1,620,628,750,804


In [34]:
# Save the clean Fact Table
fact.to_csv('Fact_Flights_Final.csv', index=False)

# Save all Dimension Tables
dim_date.to_csv('Dim_Date.csv', index=False)
dim_airport.to_csv('Dim_Airport.csv', index=False)
dim_aircraft.to_csv('Dim_Aircraft.csv', index=False)
dim_carrier.to_csv('Dim_Carrier.csv', index=False)
dim_route.to_csv('Dim_Route.csv', index=False)
dim_time.to_csv('Dim_Time.csv', index=False)

In [35]:
fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 1928368 entries, 0 to 1928367
Data columns (total 30 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Month_Name             str    
 1   Day_Name               str    
 2   FlightNum              int64  
 3   Distance               int64  
 4   Departure_Delay        float64
 5   Arrival_Delay          float64
 6   CarrierDelay           float64
 7   WeatherDelay           float64
 8   NASDelay               float64
 9   SecurityDelay          float64
 10  LateAircraftDelay      float64
 11  Delay_sum              float64
 12  ActualElapsedTime      float64
 13  Expected_ElapsedTime   float64
 14  AirTime                float64
 15  TaxiIn                 float64
 16  TaxiOut                float64
 17  Cancelled              int64  
 18  CancellationCode       str    
 19  Diverted               int64  
 20  DateKey                int64  
 21  Origin_FK              int64  
 22  Dest_FK                int64 